In [ ]:
%pip install ultralytics
%pip install paho-mqtt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 5.0 MB/s eta 0:00:00


In [ ]:
import cv2
import time
import json
import os
import sys
import numpy as np
from datetime import datetime
from ultralytics import YOLO
import paho.mqtt.client as mqtt

#Colab'in gizli kernel argümanlarını temizler, YOLO'nun çökmesini engeller
sys.argv = ['']

# 1. Edege Video Metadata & MQTT pipeline sınıfı

class EdgeVideoMetadataPipeline:
    def __init__(self, engine_path, min_aspect_ratio=0.4, max_human_pixel=254, temp_threshold=37.0,
                 mqtt_broker="broker.emqx.io", mqtt_port=1883, mqtt_topic="thermal/sentinel/alerts"):

        # INT8/Engine model yükleme
        self.model = YOLO(engine_path, task="detect")
        self.min_aspect_ratio = min_aspect_ratio
        self.max_human_pixel = max_human_pixel
        self.temp_threshold = temp_threshold
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

        # MQTT yapılandırması ve bağlantısı
        self.mqtt_broker = mqtt_broker
        self.mqtt_topic = mqtt_topic


        try:
            # eski sürümlerle uyumluluk
            try:
                self.mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
            except AttributeError:
                self.mqtt_client = mqtt.Client()
            self.mqtt_client.connect(mqtt_broker, mqtt_port, 60)
            self.mqtt_client.loop_start()
            print(f" Broker'a bağlandı: {mqtt_broker} | Topic: {mqtt_topic}")
        except Exception as e:
            print(f" Bağlantı kurulamadı: {e}")

        # Yerel log dosyası (yedek metin kaydı)
        self.log_file = "metadata_log.json"
        if os.path.exists(self.log_file):
            os.remove(self.log_file)

        self.total_json_bytes = 0

    def process_frame(self, gray_frame, frame_id):
        #1.Ön İşleme: CLAHE + Inferno renk paleti
        enhanced_gray = self.clahe.apply(gray_frame)
        inferno_frame = cv2.applyColorMap(enhanced_gray, cv2.COLORMAP_INFERNO)

        # 2. YOLO model tespiti
        results = self.model(
            enhanced_gray,
            conf=0.35,
            iou=0.45,
            verbose=False
            )[0]

        frame_detections = []

        if results.boxes is not None:
            detection_counter = 1
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                w, h = x2 - x1, y2 - y1

                # Geometri Filtresi
                if w <= 0 or h <= 0 or (h / float(w)) < self.min_aspect_ratio:
                    continue

                roi = enhanced_gray[y1:y2, x1:x2]
                if roi.size == 0 or np.max(roi) > self.max_human_pixel:
                    continue

                # Sıcaklık hesabı
                top_10 = np.percentile(roi, 90)
                avg_heat = np.mean(roi[roi >= top_10])
                temp = round(float(30.0 + (avg_heat * 0.03)), 1)
                is_anomaly = temp >= self.temp_threshold

                # B-box ve metadata hazırlığı
                det_id = f"f{frame_id}_det_{detection_counter}"
                detection_data = {
                    "detection_id": det_id,
                    "temperature_c": temp,
                    "is_anomaly": is_anomaly,
                    "bbox": [x1, y1, x2, y2]
                }
                frame_detections.append(detection_data)
                detection_counter += 1

                # Görsel Çizim (Normal: Yeşil, Anomali: Kırmızı)
                color = (0, 0, 255) if is_anomaly else (0, 255, 0)
                label = f"{temp}C" + (" [ALARM]" if is_anomaly else "")
                cv2.rectangle(inferno_frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(inferno_frame, label, (x1, max(y1-5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # 3. İnsan tespiti varsa JSON paketleme ve MQTT Publish
        # Karede en az 1 insan tespit edildiyse paketi hazırla
        if frame_detections:
            # Anomali (ateşi yüksek) olan insanları say
            anomaly_count = sum(1 for d in frame_detections if d["is_anomaly"])
            total_humans = len(frame_detections)

            payload = {
                "device_id": "thermal_edge_node_01",
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "frame_id": frame_id,
                "total_humans": total_humans,      # Toplam İnsan
                "anomaly_count": anomaly_count,    # Ateşi Yüksek İnsan
                "detections": frame_detections     # Detay Listesi
            }

            json_str = json.dumps(payload)
            self.total_json_bytes += len(json_str.encode('utf-8'))

            # A. Yerel log dosyasına yazma
            with open(self.log_file, "a") as f:
                f.write(json_str + "\n")

            # B. MQTT üzerinden FastAPI/Broker'a fırlatma
            self.mqtt_client.publish(self.mqtt_topic, json_str)

        return inferno_frame

    def close(self):
        self.mqtt_client.loop_stop()
        self.mqtt_client.disconnect()


# 2. Video çalıştırma ve test akışı

if __name__ == "__main__":
    VIDEO_PATH = "test_thermal_video.mp4"
    OUTPUT_VIDEO_PATH = "processed_thermal_video_metadata.mp4"
    ENGINE_PATH = "yolov8_gold_best_int8.engine"

    pipeline = EdgeVideoMetadataPipeline(
        engine_path=ENGINE_PATH,
        temp_threshold=37.0
    )

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print(f"Video dosyası açılamadı: {VIDEO_PATH}")
    else:
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        PLAYBACK_FPS = 5.0

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, PLAYBACK_FPS, (width, height))

        frame_count = 0
        start_time = time.time()

        print("Edge Video Pipeline çalışıyor, veriler MQTT'ye aktarılıyor...")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            #Gri tonlamaya çevir (Termal girdi kabülü)
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if len(frame.shape) == 3 else frame

            #Pipeline işleme
            processed_frame = pipeline.process_frame(gray_frame, frame_id=frame_count)

            #Anlık işleme FPS bilgilerini ekrana basma
            elapsed_time = time.time() - start_time
            current_fps = frame_count / elapsed_time if elapsed_time > 0 else 0

            cv2.putText(processed_frame, f"Edge FPS: {current_fps:.1f}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            out.write(processed_frame)

        cap.release()
        out.release()
        pipeline.close()

        print(f" Toplam {frame_count} kare işlendi.")
        print(f"Video kaydedildi: {OUTPUT_VIDEO_PATH}")
        print(f"Ortalama canlı işleme hızı {current_fps:.1f} FPS")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Broker'a bağlandı: broker.emqx.io | Topic: thermal/sentinel/alerts
Edge Video Pipeline çalışıyor, veriler MQTT'ye aktarılıyor...
Loading yolov8_gold_best_int8.engine for TensorRT inference...
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.2.0'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 251ms
Prepared 3 packages in 1m 45s
Installed 3 packages in 4ms
 + tensorrt-cu12==11.2.1.2
 + tensorrt-cu12-bindings==11.2.1.2
 + tensorrt-cu12-libs==11.2.1.2

requirements: AutoUpdate success ✅ 106.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

 Toplam 584 kare işlendi.
Video kay